# Offline inference pipeline for online-gambling comment classification.

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "./model"
TOKENIZER_DIR = "./tokenizer"
MAX_LENGTH = 128
LABEL_MAP = {0: "non_judi_online", 1: "judi_online"}

device = torch.device("cpu")
print("Inference device:", device)

In [ ]:
PHONE_PATTERN = re.compile(r'(\+62|62|0)8[1-9][0-9]{6,10}')
EMAIL_PATTERN = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
NIK_PATTERN = re.compile(r'\b\d{16}\b')
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+')
MENTION_PATTERN = re.compile(r'@\w+')
HASHTAG_PATTERN = re.compile(r'#\w+')
MULTISPACE_PATTERN = re.compile(r'\s+')

def mask_pii(text: str) -> str:
    text = EMAIL_PATTERN.sub("<EMAIL>", text)
    text = PHONE_PATTERN.sub("<PHONE>", text)
    text = NIK_PATTERN.sub("<NIK>", text)
    return text

def preprocess_text(text: str) -> str:
    text = text.lower().strip()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()
print("Local model and tokenizer loaded.")

In [ ]:
# Main prediction function with strict offline and CPU-only execution.
def predict_comment(input_text: str):
    masked = mask_pii(input_text)
    cleaned = preprocess_text(masked)

    encoded = tokenizer(
        cleaned,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits
        pred_id = int(torch.argmax(logits, dim=1).item())
        prob = torch.softmax(logits, dim=1).cpu().numpy()[0].tolist()

    return {
        "input_text": input_text,
        "masked_text": masked,
        "preprocessed_text": cleaned,
        "prediction_id": pred_id,
        "prediction_label": LABEL_MAP.get(pred_id, str(pred_id)),
        "probabilities": prob
    }

In [ ]:
sample = "Hubungi saya di 081234567890 dan cek https://contoh.com untuk jackpot"
result = predict_comment(sample)
result